<a href="https://colab.research.google.com/github/MykhailoMatsyshyn/huggingface-deep-rl-journey/blob/main/Unit%202%20-%20Introduction%20to%20Q-Learning/Unit_2_train_frozenlake_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 2: Q-Learning with FrozenLake-v1 ⛄ and Taxi-v3 🚕

## Project Overview

In this notebook, I build and train a **Reinforcement Learning agent from scratch** to master two classic environments:
1.  **FrozenLake ❄️:** Navigating a slippery grid to reach a gift without falling into ice holes.
2.  **Taxi-v3 🚖:** Autonomously picking up and dropping off passengers at the correct locations.

Instead of relying on high-level "black box" libraries, I implemented the **Q-Learning algorithm** using pure **Python and NumPy**. This project demonstrates a deep understanding of the **Bellman Equation**, **TD Learning**, and the exploration-exploitation trade-off.

<br>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="Environments"/>

---

### 🛠️ Tech Stack & Libraries

* 🐍 **Python & NumPy** – Core logic and matrix manipulation.
* 🏋️ **[Gymnasium](https://gymnasium.farama.org/)** – Standard API for RL environments.

<br>

### 🎮 Environments Solved

| Environment | Description | Link |
| :--- | :--- | :--- |
| **FrozenLake-v1** ❄️ | Navigate a slippery grid to reach the goal without falling into holes. | [Documentation](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) |
| **Taxi-v3** 🚖 | Pick up passengers and drop them off at correct locations efficiently. | [Documentation](https://gymnasium.farama.org/environments/toy_text/taxi/) |

---

## A small recap of Q-Learning

*Q-Learning* **is the RL algorithm that**:

- Trains *Q-Function*, an **action-value function** that encoded, in internal memory, by a *Q-table* **that contains all the state-action pair values.**

- Given a state and action, our Q-Function **will search the Q-table for the corresponding value.**
    
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Q function"  width="100%"/>

- When the training is done,**we have an optimal Q-Function, so an optimal Q-Table.**
    
- And if we **have an optimal Q-function**, we
have an optimal policy, since we **know for, each state, the best action to take.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Link value policy"  width="100%"/>


But, in the beginning, our **Q-Table is useless since it gives arbitrary value for each state-action pair (most of the time we initialize the Q-Table to 0 values)**. But, as we’ll explore the environment and update our Q-Table it will give us better and better approximations

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="q-learning.jpeg" width="100%"/>

This is the Q-Learning pseudocode:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


## Let's code our first Reinforcement Learning algorithm 🚀

### Install dependencies and create a virtual display 🔽

In the notebook, we'll need to generate a replay video. To do so, with Colab, **we need to have a virtual screen to render the environment** (and thus record the frames).

Hence the following cell will install the libraries and create and run a virtual screen 🖥

We’ll install multiple ones:

- `gymnasium`: Contains the FrozenLake-v1 ⛄ and Taxi-v3 🚕 environments.
- `pygame`: Used for the FrozenLake-v1 and Taxi-v3 UI.
- `numpy`: Used for handling our Q-table.

In [ ]:
!wget https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit2/requirements-unit2.txt

!sed -i 's/pyyaml==6.0/pyyaml>=6.0/g' requirements-unit2.txt
!sed -i '/pickle5/d' requirements-unit2.txt

!pip install -r requirements-unit2.txt

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

### Import the packages 📦

In addition to the installed libraries, we also use:

- `random`: To generate random numbers (that will be useful for epsilon-greedy policy).
- `imageio`: To generate a replay video.

In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle
from tqdm.notebook import tqdm

## Part 1: Frozen Lake ⛄ (non slippery version)

### Create and understand [FrozenLake environment ⛄]((https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
---

👉 https://gymnasium.farama.org/environments/toy_text/frozen_lake/

---
<br>

We're going to train our Q-Learning agent **to navigate from the starting state (S) to the goal state (G) by walking only on frozen tiles (F) and avoid holes (H)**.

We can have two sizes of environment:

- `map_name="4x4"`: a 4x4 grid version
- `map_name="8x8"`: a 8x8 grid version


The environment has two modes:

- `is_slippery=False`: The agent always moves **in the intended direction** due to the non-slippery nature of the frozen lake (deterministic).
- `is_slippery=True`: The agent **may not always move in the intended direction** due to the slippery nature of the frozen lake (stochastic).

For now let's keep it simple with the 4x4 map and non-slippery.
We add a parameter called `render_mode` that specifies how the environment should be visualised. In our case because we **want to record a video of the environment at the end, we need to set render_mode to rgb_array**.

As [explained in the documentation](https://gymnasium.farama.org/api/env/#gymnasium.Env.render) “rgb_array”: Return a single frame representing the current state of the environment. A frame is a np.ndarray with shape (x, y, 3) representing RGB values for an x-by-y pixel image.

In [ ]:
# Create the FrozenLake-v1 environment using 4x4 map and non-slippery version and render_mode="rgb_array"
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

### Let's see what the Environment looks like:


In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # Get a random observation

We see with `Observation Space Shape Discrete(16)` that the observation is an integer representing the **agent’s current position as current_row * ncols + current_col (where both the row and col start at 0)**.

For example, the goal position in the 4x4 map can be calculated as follows: 3 * 4 + 3 = 15. The number of possible observations is dependent on the size of the map. **For example, the 4x4 map has 16 possible observations.**


For instance, this is what state = 0 looks like:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

The action space (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:
- 0: GO LEFT
- 1: GO DOWN
- 2: GO RIGHT
- 3: GO UP

Reward function 💰:
- Reach goal: +1
- Reach hole: 0
- Reach frozen: 0

### Create and Initialize the Q-table 🗄️

(👀 Step 1 of the pseudocode)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


It's time to initialize our Q-table! To know how many rows (states) and columns (actions) to use, we need to know the action and observation space. We already know their values from before, but we'll want to obtain them programmatically so that our algorithm generalizes for different environments. Gym provides us a way to do that: `env.action_space.n` and `env.observation_space.n`


In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [ ]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

### Define the greedy policy 🤖

Remember we have two policies since Q-Learning is an **off-policy** algorithm. This means we're using a **different policy for acting and updating the value function**.

- Epsilon-greedy policy (acting policy)
- Greedy-policy (updating policy)

The greedy policy will also be the final policy we'll have when the Q-learning agent completes training. The greedy policy is used to select an action using the Q-table.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action = np.argmax(Qtable[state][:])

  return action

### Define the epsilon-greedy policy 🤖

Epsilon-greedy is the training policy that handles the exploration/exploitation trade-off.

The idea with epsilon-greedy:

- With *probability 1 - ɛ* : **we do exploitation** (i.e. our agent selects the action with the highest state-action pair value).

- With *probability ɛ*: we do **exploration** (trying a random action).

As the training continues, we progressively **reduce the epsilon value since we will need less and less exploration and more exploitation.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Q-Learning" width="100%"/>


In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num = random.uniform(0,1)
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action = greedy_policy(Qtable, state)
  # else --> exploration
  else:
    action = env.action_space.sample()

  return action

### Define the hyperparameters ⚙️

The exploration related hyperparamters are some of the most important ones.

- We need to make sure that our agent **explores enough of the state space** to learn a good value approximation. To do that, we need to have progressive decay of the epsilon.
- If you decrease epsilon too fast (too high decay_rate), **you take the risk that your agent will be stuck**, since your agent didn't explore enough of the state space and hence can't solve the problem.

In [ ]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7          # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"     # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate
eval_seed = []               # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0            # Exploration probability at start
min_epsilon = 0.05           # Minimum exploration probability
decay_rate = 0.0005          # Exponential decay rate for exploration prob

### Create the training loop method

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

The training loop goes like this:

```
For episode in the total of training episodes:

Reduce epsilon (since we need less and less exploration)
Reset the environment

  For step in max timesteps:    
    Choose the action At using epsilon greedy policy
    Take the action (a) and observe the outcome state(s') and reward (r)
    Update the Q-value Q(s,a) using Bellman equation Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    If done, finish the episode
    Our next state is the new state
```

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action = epsilon_greedy_policy(Qtable, state, epsilon)

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

### Train the Q-Learning agent 🏃

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

### Let's see what our Q-Learning table looks like now 👀

In [ ]:
Qtable_frozenlake

### The evaluation method 📝

- We defined the evaluation method that we're going to use to test our Q-Learning agent.

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

### Evaluate our Q-Learning agent 📈

- Usually, you should have a mean reward of 1.0
- The **environment is relatively easy** since the state space is really small (16). What you can try to do is [to replace it with the slippery version](https://gymnasium.farama.org/environments/toy_text/frozen_lake/), which introduces stochasticity, making the environment more complex.

In [ ]:
# Evaluate our Agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

### Publish our trained model to the Hub 🔥

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Generate a replay video of the agent
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
  """
  images = []
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not terminated or truncated:
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def push_to_hub(
    repo_id, model, env, video_fps=1, local_repo_path="hub"
):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat()
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  ```python

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  ```
  """

    evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

- Let's create **the model dictionary that contains the hyperparameters and the Q_table**.

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

Let's fill the `push_to_hub` function:

- `repo_id`: the name of the Hugging Face Hub Repository that will be created/updated `
(repo_id = {username}/{repo_name})`
💡 A good `repo_id` is `{username}/q-{env_id}`
- `model`: our model dictionary containing the hyperparameters and the Qtable.
- `env`: the environment.
- `commit_message`: message of the commit

In [ ]:
model

In [ ]:
username = "MykhailoMatsyshyn"
repo_name = "q-FrozenLake-v1-4x4-noSlippery"
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

## Part 2: Frozen Lake ⛄ (slippery version)

### The Real Challenge: Stochastic Environment

In the previous part, we solved the "Non-Slippery" version, which is **deterministic**: if the agent decides to go **Right**, it moves **Right** with 100% probability.

Now, we enable the **Slippery** mode (`is_slippery=True`), which introduces **stochasticity** (randomness) into the environment. This mimics the physics of a real frozen lake where you might slide in an unintended direction.

<br>

---

<br>

### Documentation & Physics

According to the official **[Gymnasium FrozenLake Documentation](https://gymnasium.farama.org/environments/toy_text/frozen_lake/):**

> "If `is_slippery=True`, the agent may not always move in the intended direction due to the slippery nature of the frozen lake (stochastic)."

**How the physics works:**
When the agent chooses an action (e.g., **Left**), there is an equal probability of:
1.  Moving **Left** (33.3%)
2.  Moving **Up** (33.3% - perpendicular)
3.  Moving **Down** (33.3% - perpendicular)

<br>

---

<br>

### ⚖️ Deterministic vs. Stochastic

| Feature | Non-Slippery (`False`) | Slippery (`True`) |
| :--- | :--- | :--- |
| **Nature** | **Deterministic** | **Stochastic** |
| **Control** | Perfect control (100%) | Partial control (approx. 33%) |
| **Strategy** | Shortest path (can walk on edges) | **Safe path** (avoids edges to prevent accidental falls) |
| **Difficulty** | Easy | Hard |

<br>

The agent must now learn a **safer policy**: it's often better to take a longer route through the center of the lake than to risk walking next to a hole where a single "slip" could be fatal.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio
import os
from tqdm.notebook import tqdm

# ==================== CUSTOM MAP ====================
custom_desc = [
    "SFFFF",
    "FFFHF",
    "FFFFF",
    "FFFHF",
    "HFFFG"
]

# ==================== VISUALIZE MAP ====================
print("🗺️  Visualizing Custom Map...")
env_visual = gym.make("FrozenLake-v1", desc=custom_desc, is_slippery=True, render_mode="rgb_array")
env_visual.reset()
img = env_visual.render()

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title("Custom 5x5 Map: The Minefield", fontsize=14, fontweight='bold')
plt.show()
env_visual.close()

# ==================== HYPERPARAMETER CONFIGURATIONS ====================
configs = [
    {
        'name': '1. Rookie (Low Training)',
        'episodes': 50000,
        'learning_rate': 0.7,
        'gamma': 0.95,
        'decay_rate': 0.005,
        'color': '#FF6B6B'
    },
    {
        'name': '2. Explorer (Slow Decay)',
        'episodes': 100000,
        'learning_rate': 0.7,
        'gamma': 0.95,
        'decay_rate': 0.00005,
        'color': '#FFA500'
    },
    {
        'name': '3. Balanced (Standard)',
        'episodes': 150000,
        'learning_rate': 0.7,
        'gamma': 0.95,
        'decay_rate': 0.0005,
        'color': '#4ECDC4'
    },
    {
        'name': '4. Master (Long Training)',
        'episodes': 200000,
        'learning_rate': 0.8,
        'gamma': 0.98,
        'decay_rate': 0.0001,
        'color': '#95E1D3'
    },
    {
        'name': '5. Ultra (Maximum)',
        'episodes': 300000,
        'learning_rate': 0.85,
        'gamma': 0.99,
        'decay_rate': 0.00005,
        'color': '#A8E6CF'
    }
]

# ==================== CUSTOM TRAINING FUNCTION (WITH PENALTY) ====================
def train_custom(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable, learning_rate, gamma):
    """Custom training function with Hole Penalty"""
    for episode in tqdm(range(n_training_episodes)):
        # Reduce epsilon
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

        state, info = env.reset()

        for step in range(max_steps):
            # Choose action
            action = epsilon_greedy_policy(Qtable, state, epsilon)

            # Take action
            new_state, reward, terminated, truncated, info = env.step(action)

            # === 🛑 ADD A PENALTY ===
            # If the episode is terminated, but the reward is 0 (we didn't reach the goal),
            # then we fell into a hole.
            if terminated and reward == 0:
                reward = -0.7
            # ==================================

            # Update Q(s,a)
            Qtable[state][action] = Qtable[state][action] + learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            # Finish episode
            if terminated or truncated:
                break

            state = new_state

    return Qtable

def evaluate_agent_custom(env, max_steps, n_eval_episodes, Q, seed):
    """
    Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
    :param env: The evaluation environment
    :param max_steps: Maximum number of steps per episode
    :param n_eval_episodes: Number of episode to evaluate the agent
    :param Q: The Q-table
    :param seed: The evaluation seed array (for taxi-v3)
    """
    episode_rewards = []
    for episode in tqdm(range(n_eval_episodes), desc="Evaluating"):
        if seed:
            state, info = env.reset(seed=seed[episode])
        else:
            state, info = env.reset()

        step = 0
        truncated = False
        terminated = False
        total_rewards_ep = 0

        for step in range(max_steps):
            # Take the action (index) that have the maximum expected future reward given that state
            action = greedy_policy(Q, state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward

            if terminated or truncated:
                break
            state = new_state

        episode_rewards.append(total_rewards_ep)

    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)

    return mean_reward, std_reward

# ==================== TRAINING & EVALUATION ====================
print("=" * 70)
print("🎯 HYPERPARAMETER COMPARISON EXPERIMENT")
print("=" * 70)

results = []
max_steps = 99
n_eval_episodes = 100

for i, config in enumerate(configs, 1):
    print(f"\n📊 Configuration {i}/5: {config['name']}")
    print(f"   Episodes: {config['episodes']:,}")
    print(f"   Learning Rate: {config['learning_rate']}")
    print(f"   Gamma: {config['gamma']}")
    print(f"   Decay Rate: {config['decay_rate']}")
    print("-" * 70)

    # Create environment
    env = gym.make("FrozenLake-v1", desc=custom_desc, is_slippery=True, render_mode="rgb_array")
    state_space = env.observation_space.n
    action_space = env.action_space.n

    # Initialize Q-table (використовуємо існуючу функцію)
    Qtable = initialize_q_table(state_space, action_space)

    # Train (використовуємо нову функцію train_custom)
    Qtable = train_custom(
        config['episodes'],
        0.05,  # min_epsilon
        1.0,   # max_epsilon
        config['decay_rate'],
        env,
        max_steps,
        Qtable,
        config['learning_rate'],  # explicit parameter
        config['gamma']           # explicit parameter
    )

    # Evaluate (використовуємо існуючу функцію)
    mean_reward, std_reward = evaluate_agent_custom(env, max_steps, n_eval_episodes, Qtable, [])

    # Store results
    results.append({
        'name': config['name'],
        'episodes': config['episodes'],
        'mean_reward': mean_reward,
        'std_reward': std_reward,
        'qtable': Qtable,
        'env': env,
        'config': config
    })

    print(f"✅ Result: {mean_reward:.3f} ± {std_reward:.3f}")

print("\n" + "=" * 70)
print("✅ All training completed!")
print("=" * 70)

# ==================== VISUALIZATION ====================

# 1. BAR CHART - Performance Comparison
print("\n📊 Creating performance comparison charts...")
df_results = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    df_results['name'],
    df_results['mean_reward'],
    yerr=df_results['std_reward'],
    capsize=5,
    color=[c['color'] for c in configs],
    alpha=0.85,
    edgecolor='black',
    linewidth=1.5
)

ax.set_title("Performance Comparison Across Configurations", fontsize=16, fontweight='bold')
ax.set_ylabel("Win Rate (0.0 - 1.0)", fontsize=13, fontweight='bold')
ax.set_xlabel("Configuration", fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_xticklabels(df_results['name'], rotation=45, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.,
        height + 0.02,
        f'{height:.3f}',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

# 2. EPISODES vs PERFORMANCE
print("📈 Creating episodes vs performance chart...")
fig, ax = plt.subplots(figsize=(10, 6))

episodes_list = [r['episodes'] for r in results]
means_list = [r['mean_reward'] for r in results]

ax.plot(episodes_list, means_list, marker='o', linewidth=2.5, markersize=10, color='#2E86AB')
ax.fill_between(episodes_list, 0, means_list, alpha=0.2, color='#2E86AB')

ax.set_title("Training Episodes vs Performance", fontsize=16, fontweight='bold')
ax.set_xlabel("Number of Training Episodes", fontsize=13, fontweight='bold')
ax.set_ylabel("Mean Reward", fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')
ax.set_ylim(0, 1.1)

# Add annotations
for x, y in zip(episodes_list, means_list):
    ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points", xytext=(0,10),
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# 3. HYPERPARAMETER IMPACT
print("🔧 Creating hyperparameter impact chart...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Learning Rate Impact
lr_values = [r['config']['learning_rate'] for r in results]
ax1.scatter(lr_values, means_list, s=150, alpha=0.7, color='#E63946', edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Learning Rate', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Reward', fontsize=12, fontweight='bold')
ax1.set_title('Learning Rate Impact', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3, linestyle='--')

# Gamma Impact
gamma_values = [r['config']['gamma'] for r in results]
ax2.scatter(gamma_values, means_list, s=150, alpha=0.7, color='#457B9D', edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Gamma (Discount Factor)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Reward', fontsize=12, fontweight='bold')
ax2.set_title('Discount Factor Impact', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# 4. SUMMARY TABLE
print("\n📋 Creating summary table...")
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')

table_data = []
for r in results:
    table_data.append([
        r['name'],
        f"{r['episodes']:,}",
        f"{r['config']['learning_rate']:.2f}",
        f"{r['config']['gamma']:.2f}",
        f"{r['config']['decay_rate']:.5f}",
        f"{r['mean_reward']:.3f} ± {r['std_reward']:.3f}"
    ])

table = ax.table(
    cellText=table_data,
    colLabels=['Configuration', 'Episodes', 'LR', 'Gamma', 'Decay', 'Performance'],
    cellLoc='center',
    loc='center',
    colWidths=[0.25, 0.15, 0.1, 0.1, 0.15, 0.25]
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(6):
    table[(0, i)].set_facecolor('#4ECDC4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color rows
colors = ['#F8F9FA', '#E9ECEF']
for i in range(1, len(table_data) + 1):
    for j in range(6):
        table[(i, j)].set_facecolor(colors[i % 2])

plt.title("Configuration Summary", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# ==================== VIDEO GENERATION ====================
print("\n🎬 Creating demonstration videos (High Quality MP4)...")

def create_winning_video(env, qtable, base_filename, max_attempts=100):
    """
    Records a video of a successful episode using the trained agent.
    Saves every step to ensure smooth animation.
    """
    filename = base_filename.replace(".gif", ".mp4")

    for attempt in range(max_attempts):
        frames = []
        state, info = env.reset(seed=None)

        frames.append(env.render())

        total_reward = 0
        steps_log = []

        for step in range(200):
            action = greedy_policy(qtable, state)
            state, reward, terminated, truncated, info = env.step(action)

            frames.append(env.render())

            total_reward += reward
            steps_log.append(action)

            if terminated or truncated:
                break

        if terminated and total_reward == 1.0:
            print(f"   🎉 Success! Recording 'Victory Lap' (Attempt #{attempt + 1})")
            print(f"      Steps taken: {len(steps_log)}")

            try:
                imageio.mimsave(filename, frames, fps=2)
                return True, filename
            except Exception as e:
                print(f"      ⚠️ Error saving MP4: {e}")
                # Fallback to GIF if MP4 fails due to codec issues
                filename = filename.replace(".mp4", ".gif")
                imageio.mimsave(filename, frames, fps=2)
                return True, filename

    print(f"   ⚠️  Agent failed to complete the map after {max_attempts} attempts for video.")
    return False, None

video_folder = "./videos_comparison"
os.makedirs(video_folder, exist_ok=True)

for i, res in enumerate(results, 1):
    name = res['name']
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "")
    base_filename = os.path.join(video_folder, f"{safe_name}.gif")

    print(f"\n🎥 Recording {i}/5: {name}")
    print(f"   Win Rate during eval: {res['mean_reward']:.3f}")

    success, saved_path = create_winning_video(res['env'], res['qtable'], base_filename)

    if success:
        print(f"   ✅ Video saved: {saved_path}")

    print("-" * 50)

# ==================== FINAL SUMMARY ====================
print("\n" + "=" * 70)
print("🎉 ANALYSIS COMPLETE!")
print("=" * 70)
print("\n🏆 Final Rankings:")

sorted_results = sorted(results, key=lambda x: x['mean_reward'], reverse=True)
for rank, res in enumerate(sorted_results, 1):
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else f"{rank}."
    print(f"{medal} {res['name']:30s} → {res['mean_reward']:.3f} ± {res['std_reward']:.3f}")

best = sorted_results[0]
print(f"\n💎 Best Configuration: {best['name']}")
print(f"   📈 Performance: {best['mean_reward']:.3f}")
print(f"   🎓 Episodes: {best['episodes']:,}")

### 🧊 FrozenLake: Experiment Analysis & Insights

#### 1. 🔑 The Real Game Changer: Environment Simplification
My initial attempts yielded a flat `0.000` success rate. The original map featured a "death trap" bottleneck (`HHHHF`) which, combined with the slippery ice mechanics, made reaching the goal statistically nearly impossible for a learning agent.

**The Breakthrough:**
I modified the environment by removing specific holes to widen the safe passage.
* **Impact:** This was the primary factor that allowed the agent to finally reach the Goal (`G`) and propagate the reward signal back to the start.
* **Lesson:** No amount of hyperparameter tuning can fix a fundamentally broken level design. The environment must be solvable before optimization can begin.

---

#### 2. 🛡️ The "Safety Net": Role of the Negative Penalty
Although the map change was the key to success, I also kept a **Reward Shaping** mechanism (a `-0.7` penalty for falling into holes) active during the successful run.

While this wasn't the primary reason for the first victory, it likely acted as a **stabilizer**:
* It reinforced the danger of the remaining holes.
* It helped the agent converge faster by explicitly marking "bad" states, preventing it from wasting time exploring the remaining death traps repeatedly.

---

#### 3. 📊 The Training Paradox: "Less is More"

Looking at the comparative performance data on the new map, the results are even more striking than expected:

* 🥇 **1. Rookie (50k episodes)** — **0.910 (Absolute Winner!)**
* 🥈 **3. Balanced (150k episodes)** — 0.810
* 🥉 **5. Ultra (300k episodes)** — 0.790
* 📉 **4. Master (200k episodes)** — 0.360 (Major Failure)

**Key Insight:** The shortest training session produced the most dominant agent (91% win rate).
In a stochastic environment like FrozenLake, "over-training" and aggressive parameters can be detrimental.
* **The Rookie** found the safe path quickly and stuck to it.
* **The Master** (High Learning Rate 0.8) performed the worst. On slippery ice, a high learning rate causes the agent to overreact to random slips, effectively "unlearning" good strategies. This led to a crash in performance (36%).

---

#### 4. 🏆 Final Verdict

1.  **Map Design is King:** The difference between 0% and 91% win rate was primarily determined by the map layout, not the code.
2.  **Early Stopping:** I could have stopped training at 50,000 episodes. The agent reached near-perfection quickly.
3.  **The Winning Formula:**
    * `Map Strategy`: **Simplified Layout** (Crucial)
    * `Learning Rate`: **0.7** (Stable)
    * `Episodes`: **50,000** (Efficient)

**Summary:** My "Rookie" agent proved that on a well-designed map, a simple model with a short training time is superior to complex configurations that overthink the problem.

## Part 3: Taxi-v3 🚖


### Create and understand [Taxi-v3 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)

<br>

---

👉 https://gymnasium.farama.org/environments/toy_text/taxi/

---

<br>

In `Taxi-v3` 🚕, there are four designated locations in the grid world indicated by R(ed), G(reen), Y(ellow), and B(lue).

When the episode starts, **the taxi starts off at a random square** and the passenger is at a random location. The taxi drives to the passenger’s location, **picks up the passenger**, drives to the passenger’s destination (another one of the four specified locations), and then **drops off the passenger**. Once the passenger is dropped off, the episode ends.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">


In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

There are **500 discrete states since there are 25 taxi positions, 5 possible locations of the passenger** (including the case when the passenger is in the taxi), and **4 destination locations.**


In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

In [ ]:
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

The action space (the set of possible actions the agent can take) is discrete with **6 actions available 🎮**:

- 0: move south
- 1: move north
- 2: move east
- 3: move west
- 4: pickup passenger
- 5: drop off passenger

Reward function 💰:

- -1 per step unless other reward is triggered.
- +20 delivering passenger.
- -10 executing “pickup” and “drop-off” actions illegally.

In [ ]:
# Create our Q table with state_size rows and action_size columns (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Q-table shape: ", Qtable_taxi .shape)

### Define the hyperparameters ⚙️

⚠ DO NOT MODIFY EVAL_SEED: the eval_seed array **allows us to evaluate your agent with the same taxi starting positions for every classmate**

In [ ]:
# Training parameters
n_training_episodes = 25000   # Total training episodes
learning_rate = 0.7           # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# DO NOT MODIFY EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148] # Evaluation seed, this ensures that all classmates agents are trained on the same taxi starting position
                                                          # Each seed has a specific starting state

# Environment parameters
env_id = "Taxi-v3"           # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05           # Minimum exploration probability
decay_rate = 0.005            # Exponential decay rate for exploration prob


### Train our Q-Learning agent 🏃

In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

### Create a model dictionary 💾 and publish our trained model to the Hub 🔥

- We create a model dictionary that will contain all the training hyperparameters for reproducibility and the Q-Table.


In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

In [ ]:
username = "MykhailoMatsyshyn"
repo_name = "q-Taxi-v3"
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

## Part 4: Load from Hub 🔽

What's amazing with Hugging Face Hub 🤗 is that you can easily load powerful models from the community.

Loading a saved model from the Hub is really easy:

1. You go https://huggingface.co/models?other=q-learning to see the list of all the q-learning saved models.
2. You select one and copy its repo_id

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/copy-id.png" alt="Copy id">

3. Then we just need to use `load_from_hub` with:
- The repo_id
- The filename: the saved model inside the repo.

In [ ]:
from urllib.error import HTTPError

from huggingface_hub import hf_hub_download


def load_from_hub(repo_id: str, filename: str) -> str:
    """
    Download a model from Hugging Face Hub.
    :param repo_id: id of the model repository from the Hugging Face Hub
    :param filename: name of the model zip file from the repository
    """
    # Get the model from the Hub, download and cache the model on your local disk
    pickle_model = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )

    with open(pickle_model, 'rb') as f:
      downloaded_model_file = pickle.load(f)

    return downloaded_model_file

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-Taxi-v3", filename="q-learning.pkl")

print(model)
env = gym.make(model["env_id"])

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

In [ ]:
from gymnasium.wrappers import RecordVideo
import glob
from IPython.display import Video

# Configuration
video_folder = "./video_taxi"
os.makedirs(video_folder, exist_ok=True)

# Initialize environment with video recording
env = gym.make(model["env_id"], render_mode="rgb_array")
env = RecordVideo(env, video_folder=video_folder, episode_trigger=lambda x: True, name_prefix="taxi-win-hunt")

# Hunt for a winning episode
n_attempts = 50
winning_episode = None

print(f"\n🕵️ Hunting for a winning run (max {n_attempts} attempts)... \n")

for episode in range(n_attempts):
    # Use random seed to find a solvent scenario
    state, info = env.reset(seed=None)
    max_steps = model["max_steps"]
    total_reward = 0

    for step in range(max_steps):
        # Greedy policy (exploitation)
        action = np.argmax(model["qtable"][state])

        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    # Check for success (passenger delivered)
    if terminated:
        print(f"🎉 Success in episode {episode}! Total Reward: {total_reward}\n")
        winning_episode = episode
        break

env.close()

# Display the winning video
if winning_episode is not None:
    video_pattern = os.path.join(video_folder, f"taxi-win-hunt-episode-{winning_episode}.mp4")
    matching_files = glob.glob(video_pattern)

    if matching_files:
        print(f"✅ Video saved: {matching_files[0]}\n")
        display(Video(matching_files[0], embed=True, width=600))
    else:
        print("❌ Video file not found.\n")
else:
    print("😢 No success found within attempt limit.\n")

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-FrozenLake-v1-no-slippery", filename="q-learning.pkl")

env = gym.make(model["env_id"], is_slippery=False)

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

In [ ]:
# Configuration
video_folder = "./video_frozenlake"
os.makedirs(video_folder, exist_ok=True)

# Initialize environment with video recording
# Note: We must pass is_slippery=False and map_name="4x4" to match the model
env = gym.make(model["env_id"], map_name="4x4", is_slippery=False, render_mode="rgb_array")
env = RecordVideo(env, video_folder=video_folder, episode_trigger=lambda x: True, name_prefix="frozenlake-win-hunt")

# Hunt for a winning episode
n_attempts = 50
winning_episode = None

print(f"\n🕵️ Hunting for a winning run (max {n_attempts} attempts)...\n")

for episode in range(n_attempts):
    # Use random seed to vary start positions/outcomes if applicable
    state, info = env.reset(seed=None)
    max_steps = model["max_steps"]
    total_reward = 0

    for step in range(max_steps):
        # Greedy policy (exploitation)
        action = np.argmax(model["qtable"][state])

        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    # Check for success (In FrozenLake, Reward 1.0 means goal reached)
    if terminated and total_reward == 1.0:
        print(f"🎉 Success in episode {episode}! Reached the goal!\n")
        winning_episode = episode
        break

env.close()

# Display the winning video
if winning_episode is not None:
    video_pattern = os.path.join(video_folder, f"frozenlake-win-hunt-episode-{winning_episode}.mp4")
    matching_files = glob.glob(video_pattern)

    if matching_files:
        print(f"✅ Video saved: {matching_files[0]}\n")
        display(Video(matching_files[0], embed=True, width=600))
    else:
        print("❌ Video file not found.\n")
else:
    print("😢 No success found within attempt limit.\n")